In [1]:
import numpy as np
import pandas as pd
import random
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

def print_metrics(y_true, y_pred, model_name="Model"):
    print(f"\n{'='*50}")
    print(f"Results for: {model_name}")
    print(f"{'='*50}")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred):.4f}")
    print(f"F1-score : {f1_score(y_true, y_pred):.4f}")
    print()
    print(classification_report(y_true, y_pred, target_names=['negative', 'positive']))

In [2]:
def load_polarity_data(neg_path='rt-polarity.neg', pos_path='rt-polarity.pos'):
    with open(neg_path, 'r', encoding='utf-8', errors='ignore') as f:
        texts_neg = f.read().splitlines()
    with open(pos_path, 'r', encoding='utf-8', errors='ignore') as f:
        texts_pos = f.read().splitlines()
    texts = texts_neg + texts_pos
    labels = [0] * len(texts_neg) + [1] * len(texts_pos)
    print(f'Negative reviews: {len(texts_neg)}')
    print(f'Positive reviews: {len(texts_pos)}')
    print(f'Total: {len(texts)}')
    return texts, labels

try:
    texts, labels = load_polarity_data()
except Exception as e:
    print(f'Auto-download failed: {e}')
    print('Please place rt-polarity.neg and rt-polarity.pos in the working directory.')
    texts, labels = load_polarity_data()

Negative reviews: 5331
Positive reviews: 5331
Total: 10662


In [3]:
print('Sample negative reviews:')
for t in texts[:3]:
    print(' -', t)

print('\nSample positive reviews:')
for t in texts[-3:]:
    print(' -', t)

Sample negative reviews:
 - simplistic , silly and tedious . 
 - it's so laddish and juvenile , only teenage boys could possibly find it funny . 
 - exploitative and largely devoid of the depth or sophistication that would make watching such a graphic treatment of the crimes bearable . 

Sample positive reviews:
 - standing in the shadows of motown is the best kind of documentary , one that makes a depleted yesterday feel very much like a brand-new tomorrow . 
 - it's nice to see piscopo again after all these years , and chaykin and headly are priceless . 
 - provides a porthole into that noble , trembling incoherence that defines us all . 


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)
print(f'Train size: {len(X_train)}')
print(f'Test size : {len(X_test)}')

Train size: 8529
Test size : 2133


TF-IDF + Logistic regression

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer(min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Vocabulary size: {len(tfidf.get_feature_names_out()):,}')

clf_tfidf = LogisticRegression(max_iter=1000, random_state=42)
clf_tfidf.fit(X_train_tfidf, y_train)

preds_tfidf = clf_tfidf.predict(X_test_tfidf)
print_metrics(y_test, preds_tfidf, model_name='TF-IDF + Logistic Regression')

Vocabulary size: 8,712

Results for: TF-IDF + Logistic Regression
Accuracy : 0.7745
Precision: 0.7676
Recall   : 0.7871
F1-score : 0.7772

              precision    recall  f1-score   support

    negative       0.78      0.76      0.77      1067
    positive       0.77      0.79      0.78      1066

    accuracy                           0.77      2133
   macro avg       0.77      0.77      0.77      2133
weighted avg       0.77      0.77      0.77      2133



In [6]:
feature_names = np.array(tfidf.get_feature_names_out())
coef = clf_tfidf.coef_[0]
sorted_idx = coef.argsort()

print('Most negative words:', list(feature_names[sorted_idx[:10]]))
print('Most positive words:', list(feature_names[sorted_idx[:-11:-1]]))

Most negative words: ['too', 'dull', 'bad', 'boring', 'worst', 'only', 'doesn', 'fails', 'so', 'tv']
Most positive words: ['and', 'still', 'funny', 'solid', 'enjoyable', 'best', 'entertaining', 'performances', 'unexpected', 'culture']


Word2Vec

In [8]:
import gensim
from gensim.models import Word2Vec
from nltk.tokenize import RegexpTokenizer

tokenizer = RegexpTokenizer(r'\w+')

def simple_tokenize(text):
    return tokenizer.tokenize(text.lower())

# Tokenize all training texts
train_tokens = [simple_tokenize(t) for t in X_train]
test_tokens  = [simple_tokenize(t) for t in X_test]

# Train Word2Vec
w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=42
)
print(f'Word2Vec vocabulary size: {len(w2v_model.wv.key_to_index):,}')

# Show similar words
for word in ['good', 'bad', 'boring']:
    if word in w2v_model.wv:
        similar = w2v_model.wv.most_similar(word, topn=5)
        print(f"\nWords similar to '{word}':")
        for w, score in similar:
            print(f'  {w}: {score:.4f}')

Word2Vec vocabulary size: 8,784

Words similar to 'good':
  bad: 0.9994
  that: 0.9991
  only: 0.9990
  because: 0.9989
  long: 0.9988

Words similar to 'bad':
  good: 0.9994
  but: 0.9993
  that: 0.9992
  so: 0.9990
  something: 0.9989

Words similar to 'boring':
  made: 0.9993
  fun: 0.9993
  great: 0.9993
  my: 0.9992
  entertaining: 0.9992


In [10]:
def document_vector(tokens, model):
    tokens_in_vocab = [w for w in tokens if w in model.wv]
    if not tokens_in_vocab:
        return np.zeros(model.vector_size)
    return np.mean([model.wv[w] for w in tokens_in_vocab], axis=0)

X_train_w2v = np.array([document_vector(t, w2v_model) for t in train_tokens])
X_test_w2v  = np.array([document_vector(t, w2v_model) for t in test_tokens])

print(f'Feature matrix shape: {X_train_w2v.shape}')

clf_w2v = LogisticRegression(max_iter=1000, random_state=42)
clf_w2v.fit(X_train_w2v, y_train)

preds_w2v = clf_w2v.predict(X_test_w2v)
print_metrics(y_test, preds_w2v)

Feature matrix shape: (8529, 100)

Results for: Model
Accuracy : 0.5767
Precision: 0.5704
Recall   : 0.6191
F1-score : 0.5938

              precision    recall  f1-score   support

    negative       0.58      0.53      0.56      1067
    positive       0.57      0.62      0.59      1066

    accuracy                           0.58      2133
   macro avg       0.58      0.58      0.58      2133
weighted avg       0.58      0.58      0.58      2133



Zero Shot

In [11]:
from transformers import pipeline

zero_shot = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli'
)
candidate_labels = ['positive', 'negative']


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [12]:
SAMPLE_SIZE = 200
X_test_sample  = X_test[:SAMPLE_SIZE]
y_test_sample  = y_test[:SAMPLE_SIZE]

preds_zs = []
for text in X_test_sample:
    result = zero_shot(text[:512], candidate_labels)
    best_label = result['labels'][0]
    preds_zs.append(1 if best_label == 'positive' else 0)

print_metrics(y_test_sample, preds_zs, model_name=f'Zero-Shot (bart-large-mnli) on {SAMPLE_SIZE} samples')


Results for: Zero-Shot (bart-large-mnli) on 200 samples
Accuracy : 0.8100
Precision: 0.8734
Recall   : 0.7113
F1-score : 0.7841

              precision    recall  f1-score   support

    negative       0.77      0.90      0.83       103
    positive       0.87      0.71      0.78        97

    accuracy                           0.81       200
   macro avg       0.82      0.81      0.81       200
weighted avg       0.82      0.81      0.81       200

